# MLOps体験

合成データからスポーツへの関心ラベルを予測します。実在する人の属性や、誰が視聴していたかを特定するものではありません。

学習 → 評価 → モデル登録 → 登録したモデルで予測 → 結果保存の順に進みます。

## 1. 実行前の確認

第2章の6回のdbt buildを完了してから実行します。SnowsightのWorkspace NotebookでPythonランタイムを選び、`BCAST_PLATFORM_ENGINEER_ROLE`、`BCAST_PLATFORM_COMMON_WH`を使います。必要なパッケージは pandas、scikit-learn、snowflake-ml-python、snowflake-snowpark-pythonです。

このNotebookはローカルPCでは実行しません。モデルV1が既にある場合は登録前に停止します。別版を試す場合はMODEL_VERSIONをV2などに変更して上から実行します。既存モデルを削除しません。予測結果テーブルは最後の保存セルで置き換えます。

In [ ]:
import re
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as sf
from snowflake.snowpark.types import StructType, StructField, StringType, LongType
from snowflake.ml.registry import Registry

session = get_active_session()
if session.get_current_role().strip('"') != 'BCAST_PLATFORM_ENGINEER_ROLE':
    raise RuntimeError('NotebookのロールをBCAST_PLATFORM_ENGINEER_ROLEに変更してください。')
session.use_warehouse('BCAST_PLATFORM_COMMON_WH')
session.use_database('BCAST_PLATFORM_HANDSON')
session.use_schema('ML')
MODEL_NAME = 'SPORTS_INTEREST_MODEL'
MODEL_VERSION = 'V1'
if not re.fullmatch(r'V[1-9][0-9]*', MODEL_VERSION):
    raise ValueError('MODEL_VERSIONはV1、V2などを指定します。')
FEATURE_COLUMNS = ['NEWS_SHARE', 'DRAMA_SHARE', 'VARIETY_SHARE', 'ANIME_SHARE', 'SPORTS_SHARE', 'TOTAL_MINUTES', 'TOTAL_SESSIONS']
print('scikit-learn', sklearn.__version__)
print('登録予定:', MODEL_NAME, MODEL_VERSION)

## 2. 1台につき1行のヒントを作る

5局の共通マートからジャンル別の視聴時間割合を作ります。集計は共通WHで行い、Notebookに持ってくるのは最大201行だけです。200台の教材から増えていたら停止します。

端末ID・ラベルが分かるかどうか・正解ラベルは、モデルへの入力に使いません。

In [ ]:
daily = session.table('BCAST_PLATFORM_HANDSON.COMMON.VIEWING_DAILY')
features_sdf = daily.group_by('DEVICE_ID').agg(
    sf.sum('VIEW_MINUTES').cast('double').alias('TOTAL_MINUTES'),
    sf.sum('SESSION_COUNT').cast('double').alias('TOTAL_SESSIONS'),
    *[sf.sum(sf.when(sf.col('GENRE') == genre, sf.col('VIEW_MINUTES')).otherwise(sf.lit(0))).cast('double').alias(genre + '_MINUTES')
      for genre in ['NEWS', 'DRAMA', 'VARIETY', 'ANIME', 'SPORTS']]
)
for genre in ['NEWS', 'DRAMA', 'VARIETY', 'ANIME', 'SPORTS']:
    features_sdf = features_sdf.with_column(genre + '_SHARE', sf.when(sf.col('TOTAL_MINUTES') > 0, sf.col(genre + '_MINUTES') / sf.col('TOTAL_MINUTES')).otherwise(sf.lit(None)))
features_pdf = features_sdf.select('DEVICE_ID', *FEATURE_COLUMNS).sort('DEVICE_ID').limit(201).to_pandas()
labels_pdf = session.table('BCAST_PLATFORM_HANDSON.RAW.DEVICE_LABELS').select('DEVICE_ID', 'LABEL_AVAILABLE', 'TARGET_SPORTS_FAN').limit(201).to_pandas()
if len(features_pdf) != 200 or len(labels_pdf) != 200:
    raise ValueError('教材は200台です。第1・2章のデータを確認してください。')
if features_pdf.DEVICE_ID.duplicated().any() or labels_pdf.DEVICE_ID.duplicated().any():
    raise ValueError('端末IDが重複しています。')
expected_devices = {f'D{device_number:04d}' for device_number in range(1, 201)}
if features_pdf.DEVICE_ID.isna().any() or labels_pdf.DEVICE_ID.isna().any():
    raise ValueError('端末IDに欠損があります。')
if set(features_pdf.DEVICE_ID) != expected_devices or set(labels_pdf.DEVICE_ID) != expected_devices:
    raise ValueError('端末IDはD0001からD0200までが必要です。')
features_pdf[FEATURE_COLUMNS] = features_pdf[FEATURE_COLUMNS].astype('float64')
if not np.isfinite(features_pdf[FEATURE_COLUMNS].to_numpy()).all():
    raise ValueError('特徴量に欠損または無限大があります。')
if labels_pdf.LABEL_AVAILABLE.isna().any():
    raise ValueError('ラベル公開フラグに欠損があります。')
known_mask = labels_pdf.LABEL_AVAILABLE.eq(True)
if known_mask.sum() != 100 or labels_pdf.loc[~known_mask, 'TARGET_SPORTS_FAN'].notna().any():
    raise ValueError('既知100台・未知100台（正解NULL）の契約に一致しません。')
known_pdf = features_pdf.merge(labels_pdf.loc[known_mask, ['DEVICE_ID', 'TARGET_SPORTS_FAN']], on='DEVICE_ID', validate='one_to_one')
if known_pdf.TARGET_SPORTS_FAN.isna().any() or not known_pdf.TARGET_SPORTS_FAN.isin([0, 1]).all():
    raise ValueError('既知ラベルは0/1が必要です。')
if known_pdf.TARGET_SPORTS_FAN.value_counts().to_dict() != {0: 50, 1: 50}:
    raise ValueError('教材の既知ラベルは各クラス50台です。')
print('全端末:', len(features_pdf), '既知ラベル:', len(known_pdf))
features_pdf.head()

## 3. 学習用80台・採点用20台に分ける

浅い決定木を1本だけ学習します。採点用の端末は学習に混ぜません。同じ採点データを見ながら設定を調整し続けると過大評価になるため、ここでは設定を固定します。比較用に「多数派を答えるだけ」のモデルも採点します。

In [ ]:
train_pdf, evaluation_pdf = train_test_split(known_pdf, test_size=0.2, random_state=42, stratify=known_pdf['TARGET_SPORTS_FAN'])
assert set(train_pdf.DEVICE_ID).isdisjoint(set(evaluation_pdf.DEVICE_ID))
train_features = train_pdf[FEATURE_COLUMNS]
evaluation_features = evaluation_pdf[FEATURE_COLUMNS]
train_labels = train_pdf['TARGET_SPORTS_FAN'].astype('int64')
evaluation_labels = evaluation_pdf['TARGET_SPORTS_FAN'].astype('int64')
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=42)
model.fit(train_features, train_labels)
baseline = DummyClassifier(strategy='most_frequent').fit(train_features, train_labels)
evaluation_predictions = model.predict(evaluation_features)
heldout_accuracy = float(accuracy_score(evaluation_labels, evaluation_predictions))
baseline_accuracy = float(accuracy_score(evaluation_labels, baseline.predict(evaluation_features)))
pd.DataFrame([
    {'MODEL': 'Decision tree', 'ACCURACY': heldout_accuracy, 'BALANCED_ACCURACY': balanced_accuracy_score(evaluation_labels, evaluation_predictions)},
    {'MODEL': 'Majority baseline', 'ACCURACY': baseline_accuracy, 'BALANCED_ACCURACY': 0.5}
])

## 4. モデルに名前と版を付けて登録する

この後はSnowflakeにモデルを書き込みます。採点したモデルそのものを登録し、全件で学習し直しません。既存の同名・同版がある場合は停止します。続けるにはMODEL_VERSIONを別の版へ変更し、先頭から実行してください。

登録処理や依存パッケージ解決が失敗した場合は停止し、講師に相談してください。コンテナで学習しても、予測の実行先は明示的にWAREHOUSEを指定します。

In [ ]:
prediction_model = None
registered_version = None
registry = Registry(session=session, database_name='BCAST_PLATFORM_HANDSON', schema_name='ML')
existing_models = registry.show_models()
existing_models.columns = [str(column).upper() for column in existing_models.columns]
if not existing_models.empty:
    if 'NAME' not in existing_models.columns:
        raise RuntimeError('モデル一覧の列を確認してください。安全のため停止します。')
    if existing_models['NAME'].str.upper().eq(MODEL_NAME).any():
        existing_versions = registry.get_model(MODEL_NAME).show_versions()
        existing_versions.columns = [str(column).upper() for column in existing_versions.columns]
        if 'NAME' not in existing_versions.columns:
            raise RuntimeError('モデル版一覧の列を確認してください。安全のため停止します。')
        if existing_versions['NAME'].str.upper().eq(MODEL_VERSION).any():
            raise RuntimeError('同じ版が存在します。MODEL_VERSIONを別の版へ変更して先頭から実行してください。')
registered_version = registry.log_model(
    model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    sample_input_data=train_features.head(10),
    conda_dependencies=['scikit-learn==' + sklearn.__version__],
    target_platforms=['WAREHOUSE'],
    options={'relax_version': False},
    metrics={'heldout_accuracy': heldout_accuracy, 'baseline_accuracy': baseline_accuracy},
    comment='Synthetic educational sports-interest classification. Not real demographic or viewer identification.'
)
registered_name = registered_version.model_name
registered_version_name = registered_version.version_name
print('登録:', registered_name, registered_version_name)
registered_version.show_functions()

## 5. 登録したモデルで200台を予測する

登録済みモデルを呼び出します。入力列を保持して返すSnowpark入力を使い、計算は共通WHで実行します。戻り値の行順だけに端末IDを依存させず、戻り値に残った入力特徴量で突き合わせます。同じ特徴量の端末は同じ予測になります。

予測には学習用80台も含みますが、ここでは精度を測りません。精度の評価は先ほどの20台だけです。

In [ ]:
prediction_model = None
prediction_input = features_pdf[FEATURE_COLUMNS].drop_duplicates().reset_index(drop=True)
from snowflake.snowpark.types import DoubleType
prediction_schema = StructType([StructField(column, DoubleType()) for column in FEATURE_COLUMNS])
prediction_sdf = session.create_dataframe(prediction_input.to_numpy().tolist(), schema=prediction_schema)
registry_result = registered_version.run(prediction_sdf, function_name='predict')
registry_output = registry_result.limit(201).to_pandas()
if not isinstance(registry_output, pd.DataFrame):
    raise RuntimeError('推論結果を表へ変換できません。')
registry_output.columns = [str(column).upper() for column in registry_output.columns]
if not set(FEATURE_COLUMNS).issubset(registry_output.columns):
    raise RuntimeError('推論結果に入力列が保持されていません。ID対応を確認するまで保存しません。')
output_columns = [column for column in registry_output.columns if column not in FEATURE_COLUMNS]
if len(output_columns) != 1:
    raise RuntimeError('predictの出力列を一つに特定できません: ' + str(output_columns))
registry_output = registry_output.rename(columns={output_columns[0]: 'PREDICTED_SPORTS_FAN'})
if len(registry_output) != len(prediction_input) or registry_output.duplicated(FEATURE_COLUMNS).any():
    raise ValueError('推論行数または特徴量の一意性が一致しません。')
if registry_output['PREDICTED_SPORTS_FAN'].isna().any() or not registry_output['PREDICTED_SPORTS_FAN'].isin([0, 1]).all():
    raise ValueError('予測は欠損なしの0/1が必要です。')
prediction_pdf = features_pdf.merge(registry_output, on=FEATURE_COLUMNS, how='left', validate='many_to_one')
if len(prediction_pdf) != 200 or prediction_pdf['PREDICTED_SPORTS_FAN'].isna().any():
    raise ValueError('全端末への予測対応を確認できません。保存しません。')
prediction_model = (registered_version.model_name, registered_version.version_name)
prediction_pdf[['DEVICE_ID', 'PREDICTED_SPORTS_FAN']].head()

## 6. 結果と使用したモデルの版を保存する

次のセルは `BCAST_PLATFORM_HANDSON.ML.PREDICTIONS` の内容を置き換えます。保存を意図した場合だけ `SAVE_RESULTS = True` にして実行してください。モデルや元のログは削除しません。

Streamlitではこれを「合成データの推定関心」として表示します。予測日とモデルの版を残すことで、どのモデルが出した結果なのかを追えるようになります。

In [ ]:
SAVE_RESULTS = False
if not SAVE_RESULTS:
    raise RuntimeError('結果を保存する場合はSAVE_RESULTSをTrueに変更してください。')
if globals().get('prediction_model') != (registered_name, registered_version_name) or (MODEL_NAME, MODEL_VERSION) != (registered_name, registered_version_name):
    raise ValueError('保存前確認: このモデル版で推論を完了してください。')
if len(prediction_pdf) != 200 or prediction_pdf.DEVICE_ID.isna().any() or prediction_pdf.DEVICE_ID.duplicated().any():
    raise ValueError('保存前確認: 端末は欠損・重複なしの200台が必要です。')
if set(prediction_pdf.DEVICE_ID) != {f'D{device_number:04d}' for device_number in range(1, 201)}:
    raise ValueError('保存前確認: 端末IDが一致しません。')
if prediction_pdf.PREDICTED_SPORTS_FAN.isna().any() or not prediction_pdf.PREDICTED_SPORTS_FAN.isin([0, 1]).all():
    raise ValueError('保存前確認: 予測は0/1が必要です。')
if registered_name != registered_version.model_name or registered_version_name != registered_version.version_name:
    raise ValueError('保存前確認: モデルの版が一致しません。')
output_rows = [(str(row.DEVICE_ID), int(row.PREDICTED_SPORTS_FAN)) for row in prediction_pdf.itertuples(index=False)]
output_schema = StructType([StructField('DEVICE_ID', StringType()), StructField('PREDICTED_SPORTS_FAN', LongType())])
output_sdf = session.create_dataframe(output_rows, schema=output_schema).with_column('MODEL_NAME', sf.lit(registered_name)).with_column('MODEL_VERSION', sf.lit(registered_version_name)).with_column('PREDICTED_AT', sf.convert_timezone(sf.lit('UTC'), sf.current_timestamp()).cast('timestamp_ntz'))
output_sdf.write.mode('overwrite').save_as_table('BCAST_PLATFORM_HANDSON.ML.PREDICTIONS')
saved_pdf = session.table('BCAST_PLATFORM_HANDSON.ML.PREDICTIONS').limit(201).to_pandas()
if len(saved_pdf) != 200 or saved_pdf.DEVICE_ID.duplicated().any():
    raise RuntimeError('保存後確認に失敗しました。後続に進まないでください。')
expected_saved = prediction_pdf[['DEVICE_ID', 'PREDICTED_SPORTS_FAN']].sort_values('DEVICE_ID').reset_index(drop=True)
actual_saved = saved_pdf[['DEVICE_ID', 'PREDICTED_SPORTS_FAN']].sort_values('DEVICE_ID').reset_index(drop=True)
pd.testing.assert_frame_equal(expected_saved, actual_saved, check_dtype=False)
if not saved_pdf.MODEL_NAME.eq(registered_name).all() or not saved_pdf.MODEL_VERSION.eq(registered_version_name).all() or saved_pdf.PREDICTED_AT.isna().any():
    raise RuntimeError('保存後のモデル版または保存日時が一致しません。')
print('200台の結果を保存・照合しました。PREDICTED_ATはUTCの保存時刻です。次の説明で出力の記録とNotebookサービスの停止を確認してから、第4章へ進みます。')

## 7. できたこと・やっていないこと

- 学習と採点を分け、比較用の基準とともにモデルを評価しました。
- 名前と版を付けてモデルを登録し、その登録モデルを使って予測しました。
- 結果にモデル版を残しました。これが今回のMLOpsの入口です。
- 自動再学習、ドリフト監視、本番精度の保証はこのNotebookに含みません。
- 実行中にパッケージ・権限・戻り値のエラーが出た場合は、後続セルを飛ばして進めず講師へ報告します。
- 保存・照合まで成功したら、評価値・モデル版・必要な出力を記録します。
- 続いて **Connected → サービス名 → Suspend** で自分のNotebookサービスを停止し、**SUSPENDED** を確認します。ブラウザーを閉じるだけでは停止しません。同じサービスにつながる別のNotebookも切断され、変数や追加パッケージが失われるため、別作業が動いていないことを先に確認します。
- 詳細は[第3章の停止手順](../docs/03_mlops.md#8-notebookの実行サービスを停止する)を確認してください。共有compute poolは停止・削除しません。
- サービス停止を確認してから、第4章のStreamlitで保存済みの結果を確認します。